In [29]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import joblib

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [30]:
# ==========================================================
# Load Dataset
# ==========================================================

orders = pd.read_csv("datasets/processed_orders.csv")

print("Orders Shape :", orders.shape)

orders.head()

Orders Shape : (5000, 34)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Sales per Unit,Cost per Unit,Distance Category,High Value Order,Route ID,Shipping Rate ($/km),Estimated Shipping Cost,Delivery Efficiency Score,Factory Order Count,Factory Workload
0,1,ORD00001,2025-02-07,2025-02-08,1,CUST9313,USA,Houston,TX,38140,...,2.69,1.74,2,1,Sugar Shack_South,1.1,326.7,297.0,1676,0
1,2,ORD00002,2025-05-04,2025-05-05,1,CUST3028,USA,Chicago,IL,39260,...,3.56,2.88,2,0,Lot's O' Nuts_Central,1.1,424.6,386.0,996,0
2,3,ORD00003,2025-04-24,2025-04-25,1,CUST7867,USA,New York,NY,28907,...,3.70,2.89,2,0,Secret Factory_East,1.1,333.3,303.0,977,2
3,4,ORD00004,2025-10-25,2025-10-29,2,CUST2028,USA,Los Angeles,CA,83972,...,3.72,2.43,1,0,Lot's O' Nuts_West,0.6,313.2,130.5,996,0
4,5,ORD00005,2025-08-07,2025-08-12,1,CUST6924,USA,Houston,TX,49291,...,4.67,2.66,3,1,Sugar Shack_South,1.1,2504.7,455.4,1676,0


In [31]:
# ==========================================================
# Factory List
# ==========================================================

factories = [

    "Lot's O' Nuts",

    "Wicked Choccy's",

    "Sugar Shack",

    "Secret Factory",

    "The Other Factory"

]

factories

["Lot's O' Nuts",
 "Wicked Choccy's",
 'Sugar Shack',
 'Secret Factory',
 'The Other Factory']

In [32]:
# ==========================================================
# Factory List
# ==========================================================

factories = [

    "Lot's O' Nuts",

    "Wicked Choccy's",

    "Sugar Shack",

    "Secret Factory",

    "The Other Factory"

]

factories

["Lot's O' Nuts",
 "Wicked Choccy's",
 'Sugar Shack',
 'Secret Factory',
 'The Other Factory']

In [33]:
# ==========================================================
# Create Simulation Dataset
# ==========================================================

simulation_rows = []

for _, row in orders.iterrows():

    for factory in factories:

        new_row = row.copy()

        new_row["Current Factory"] = row["Factory"]

        new_row["Simulated Factory"] = factory

        simulation_rows.append(new_row)

simulation_df = pd.DataFrame(simulation_rows)

print("Simulation Dataset Shape :", simulation_df.shape)

simulation_df.head()

Simulation Dataset Shape : (25000, 36)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Distance Category,High Value Order,Route ID,Shipping Rate ($/km),Estimated Shipping Cost,Delivery Efficiency Score,Factory Order Count,Factory Workload,Current Factory,Simulated Factory
0,1,ORD00001,2025-02-07,2025-02-08,1,CUST9313,USA,Houston,TX,38140,...,2,1,Sugar Shack_South,1.1,326.7,297.0,1676,0,2,Lot's O' Nuts
0,1,ORD00001,2025-02-07,2025-02-08,1,CUST9313,USA,Houston,TX,38140,...,2,1,Sugar Shack_South,1.1,326.7,297.0,1676,0,2,Wicked Choccy's
0,1,ORD00001,2025-02-07,2025-02-08,1,CUST9313,USA,Houston,TX,38140,...,2,1,Sugar Shack_South,1.1,326.7,297.0,1676,0,2,Sugar Shack
0,1,ORD00001,2025-02-07,2025-02-08,1,CUST9313,USA,Houston,TX,38140,...,2,1,Sugar Shack_South,1.1,326.7,297.0,1676,0,2,Secret Factory
0,1,ORD00001,2025-02-07,2025-02-08,1,CUST9313,USA,Houston,TX,38140,...,2,1,Sugar Shack_South,1.1,326.7,297.0,1676,0,2,The Other Factory


In [34]:
# ==========================================================
# Distance Adjustment Rules
# ==========================================================

distance_factor = {

    "Lot's O' Nuts":0.90,

    "Wicked Choccy's":0.95,

    "Sugar Shack":1.00,

    "Secret Factory":0.85,

    "The Other Factory":1.10

}

In [35]:
# ==========================================================
# Simulated Distance
# ==========================================================

simulation_df["Simulated Distance (km)"] = (

    simulation_df["Distance (km)"] *

    simulation_df["Simulated Factory"].map(distance_factor)

).round(2)

simulation_df[[
    "Distance (km)",
    "Simulated Factory",
    "Simulated Distance (km)"
]].head()

,Distance (km),Simulated Factory,Simulated Distance (km)
0,297,Lot's O' Nuts,267.30
0,297,Wicked Choccy's,282.15
0,297,Sugar Shack,297.00
0,297,Secret Factory,252.45
0,297,The Other Factory,326.70


In [36]:
# ==========================================================
# Predicted Shipping Cost
# ==========================================================

simulation_df["Predicted Shipping Cost"] = (

    simulation_df["Simulated Distance (km)"] *

    simulation_df["Shipping Rate ($/km)"]

).round(2)

In [37]:
feature_columns = [

    "Distance (km)",

    "Sales",

    "Units",

    "Cost",

    "Estimated Shipping Cost",

    "Ship Mode",

    "Division",

    "Region",

    "Factory"

]

In [38]:
prediction_df = simulation_df.copy()

In [39]:
# ==========================================================
# Prepare Prediction Data
# ==========================================================

prediction_df = simulation_df.copy()

# Replace with simulated values
prediction_df["Distance (km)"] = prediction_df["Simulated Distance (km)"]
prediction_df["Estimated Shipping Cost"] = prediction_df["Predicted Shipping Cost"]

In [40]:
# Use the simulated factory for prediction
prediction_df["Factory"] = prediction_df["Simulated Factory"]

In [41]:
# ==========================================================
# Encode Categorical Columns
# ==========================================================

from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    "Ship Mode",
    "Division",
    "Region",
    "Factory"
]

encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    prediction_df[col] = le.fit_transform(prediction_df[col])

    encoders[col] = le

print("Encoding Complete")

Encoding Complete


In [42]:
# ==========================================================
# Predict Lead Time
# ==========================================================

feature_columns = [

    "Distance (km)",
    "Sales",
    "Units",
    "Cost",
    "Estimated Shipping Cost",
    "Ship Mode",
    "Division",
    "Region",
    "Factory"

]

simulation_df["Predicted Lead Time"] = model.predict(
    prediction_df[feature_columns]
)

simulation_df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Shipping Rate ($/km),Estimated Shipping Cost,Delivery Efficiency Score,Factory Order Count,Factory Workload,Current Factory,Simulated Factory,Simulated Distance (km),Predicted Shipping Cost,Predicted Lead Time
0,1,ORD00001,2025-02-07,2025-02-08,1,CUST9313,USA,Houston,TX,38140,...,1.1,326.7,297.0,1676,0,2,Lot's O' Nuts,267.30,294.03,1.002243
0,1,ORD00001,2025-02-07,2025-02-08,1,CUST9313,USA,Houston,TX,38140,...,1.1,326.7,297.0,1676,0,2,Wicked Choccy's,282.15,310.36,1.002243
0,1,ORD00001,2025-02-07,2025-02-08,1,CUST9313,USA,Houston,TX,38140,...,1.1,326.7,297.0,1676,0,2,Sugar Shack,297.00,326.70,1.002243
0,1,ORD00001,2025-02-07,2025-02-08,1,CUST9313,USA,Houston,TX,38140,...,1.1,326.7,297.0,1676,0,2,Secret Factory,252.45,277.70,1.002243
0,1,ORD00001,2025-02-07,2025-02-08,1,CUST9313,USA,Houston,TX,38140,...,1.1,326.7,297.0,1676,0,2,The Other Factory,326.70,359.37,1.002243


In [27]:
print(type(model))

<class 'sklearn.ensemble._gb.GradientBoostingRegressor'>


In [28]:
import joblib

model = joblib.load("best_lead_time_model.pkl")

print(model)

GradientBoostingRegressor(random_state=42)


In [43]:
# ==========================================================
# Lead Time Improvement
# ==========================================================

simulation_df["Lead Time Improvement"] = (
    simulation_df["Lead Time"] -
    simulation_df["Predicted Lead Time"]
).round(2)

In [44]:
# ==========================================================
# Shipping Cost Savings
# ==========================================================

simulation_df["Shipping Cost Savings"] = (
    simulation_df["Estimated Shipping Cost"] -
    simulation_df["Predicted Shipping Cost"]
).round(2)

In [45]:
# ==========================================================
# Estimated Gross Profit
# ==========================================================

simulation_df["Estimated Gross Profit"] = (
    simulation_df["Sales"]
    - simulation_df["Cost"]
    - simulation_df["Predicted Shipping Cost"]
).round(2)

In [46]:
# ==========================================================
# Profit Impact
# ==========================================================

simulation_df["Profit Impact"] = (
    simulation_df["Estimated Gross Profit"]
    - simulation_df["Gross Profit"]
).round(2)

In [47]:
# ==========================================================
# Optimization Score
# ==========================================================

simulation_df["Optimization Score"] = (

    (simulation_df["Lead Time Improvement"] * 0.50)

    +

    (simulation_df["Shipping Cost Savings"] * 0.30)

    +

    (simulation_df["Profit Impact"] * 0.20)

).round(2)

In [48]:
# ==========================================================
# Rank Factory Options
# ==========================================================

simulation_df = simulation_df.sort_values(

    by="Optimization Score",

    ascending=False

)

best_factory = (

    simulation_df
    .groupby("Order ID")
    .first()
    .reset_index()

)

best_factory.head()

,Order ID,Row ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Current Factory,Simulated Factory,Simulated Distance (km),Predicted Shipping Cost,Predicted Lead Time,Lead Time Improvement,Shipping Cost Savings,Estimated Gross Profit,Profit Impact,Optimization Score
0,ORD00001,1,2025-02-07,2025-02-08,1,CUST9313,USA,Houston,TX,38140,...,2,Secret Factory,252.45,277.70,1.002243,-0.0,49.00,-6.80,-277.70,-40.84
1,ORD00002,2,2025-05-04,2025-05-05,1,CUST3028,USA,Chicago,IL,39260,...,0,Secret Factory,328.10,360.91,1.002243,-0.0,63.69,-322.96,-360.91,-53.08
2,ORD00003,3,2025-04-24,2025-04-25,1,CUST7867,USA,New York,NY,28907,...,1,Secret Factory,257.55,283.31,1.002243,-0.0,49.99,-256.60,-283.31,-41.67
3,ORD00004,4,2025-10-25,2025-10-29,2,CUST2028,USA,Los Angeles,CA,83972,...,0,Secret Factory,443.70,266.22,2.997709,1.0,46.98,-129.99,-266.22,-38.65
4,ORD00005,5,2025-08-07,2025-08-12,1,CUST6924,USA,Houston,TX,49291,...,2,Secret Factory,1935.45,2129.00,4.001485,1.0,375.70,-1787.50,-2129.00,-312.59


In [49]:
# ==========================================================
# Recommendation
# ==========================================================

best_factory["Recommendation"] = np.where(

    best_factory["Current Factory"] ==
    best_factory["Simulated Factory"],

    "Keep Current",

    "Reallocate"

)

In [50]:
# ==========================================================
# Priority Level
# ==========================================================

best_factory["Priority"] = np.select(

    [

        best_factory["Lead Time Improvement"] >= 2,

        best_factory["Lead Time Improvement"] >= 1

    ],

    [

        "High",

        "Medium"

    ],

    default="Low"

)

In [51]:
# ==========================================================
# Scenario Confidence Score
# ==========================================================

best_factory["Scenario Confidence Score"] = (

    75

    + best_factory["Lead Time Improvement"].clip(lower=0) * 5

).clip(upper=100).round(1)

In [52]:
# ==========================================================
# Save Recommendation Dataset
# ==========================================================

best_factory.to_csv(
    "datasets/factory_recommendations.csv",
    index=False
)

print("factory_recommendations.csv created successfully!")

best_factory.head()

factory_recommendations.csv created successfully!


,Order ID,Row ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Predicted Shipping Cost,Predicted Lead Time,Lead Time Improvement,Shipping Cost Savings,Estimated Gross Profit,Profit Impact,Optimization Score,Recommendation,Priority,Scenario Confidence Score
0,ORD00001,1,2025-02-07,2025-02-08,1,CUST9313,USA,Houston,TX,38140,...,277.70,1.002243,-0.0,49.00,-6.80,-277.70,-40.84,Reallocate,Low,75.0
1,ORD00002,2,2025-05-04,2025-05-05,1,CUST3028,USA,Chicago,IL,39260,...,360.91,1.002243,-0.0,63.69,-322.96,-360.91,-53.08,Reallocate,Low,75.0
2,ORD00003,3,2025-04-24,2025-04-25,1,CUST7867,USA,New York,NY,28907,...,283.31,1.002243,-0.0,49.99,-256.60,-283.31,-41.67,Reallocate,Low,75.0
3,ORD00004,4,2025-10-25,2025-10-29,2,CUST2028,USA,Los Angeles,CA,83972,...,266.22,2.997709,1.0,46.98,-129.99,-266.22,-38.65,Reallocate,Medium,80.0
4,ORD00005,5,2025-08-07,2025-08-12,1,CUST6924,USA,Houston,TX,49291,...,2129.00,4.001485,1.0,375.70,-1787.50,-2129.00,-312.59,Reallocate,Medium,80.0


In [53]:
print(best_factory.shape)

print(best_factory["Recommendation"].value_counts())

print(best_factory["Priority"].value_counts())

(5000, 47)
Recommendation
Reallocate    5000
Name: count, dtype: int64
Priority
Low       3752
Medium    1248
Name: count, dtype: int64
